In [4]:
import pandas as pd
from pathlib import Path
df = pd.read_csv("../tabular_4/tabular_4.csv")

print(df.shape)
print(df.columns.tolist())



(100000, 17)
['year', 'gender', 'age', 'location', 'race:AfricanAmerican', 'race:Asian', 'race:Caucasian', 'race:Hispanic', 'race:Other', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'hbA1c_level', 'blood_glucose_level', 'diabetes', 'clinical_notes']


In [ ]:
missing_summary = df.isnull().sum()

# Print columns with missing values (display only those with missing values)
missing_cols = missing_summary[missing_summary > 0]
print("Total number of missing columns:", len(missing_cols))
display(missing_cols)

# Proportion of missing data
missing_ratio = (df.isnull().mean() * 100).round(2)
print("\n Proportion of missing values（%）Top 10 columns：")
display(missing_ratio.sort_values(ascending=False).head(10))

Total number of missing columns: 0


Series([], dtype: int64)


 Proportion of missing values（%）Top 10 columns：


year                   0.0
hypertension           0.0
diabetes               0.0
blood_glucose_level    0.0
hbA1c_level            0.0
bmi                    0.0
smoking_history        0.0
heart_disease          0.0
race:Other             0.0
gender                 0.0
dtype: float64

In [19]:
data_path = Path("../tabular_4/tabular_4.csv")
out_dir = Path("../tabular_4")
out_dir.mkdir(parents=True, exist_ok=True)

# Template definition
template_features = [
    "Age", "Sex", "BMI", "GenHlth",
    "HighBP", "DiffWalk", "HighChol", "HeartDiseaseorAttack"
]
label_col = "Diabetes_012"

mapping = {
    "age": "Age",
    "gender": "Sex",
    "bmi": "BMI",
    "hypertension": "HighBP",
    "heart_disease": "HeartDiseaseorAttack",
    "diabetes": "Diabetes_012"
}

df = pd.read_csv(data_path)

aligned_features = pd.DataFrame(index=df.index)
aligned_masks    = pd.DataFrame(index=df.index)

for feat in template_features:
    src_cols = [k for k, v in mapping.items() if v == feat]
    if src_cols:
        src = src_cols[0]
        aligned_features[feat] = df[src].fillna(0)
        aligned_masks[f"{feat}_mask"] = df[src].notna().astype("int8")
    else:
        aligned_features[feat] = 0
        aligned_masks[f"{feat}_mask"] = pd.Series(0, index=df.index, dtype="int8")

label_src = None
for k, v in mapping.items():
    if v == label_col:
        label_src = k
        break

if label_src in df.columns:
    label_series = df[label_src].fillna(0)
else:
    label_series = pd.Series(0, index=df.index)

aligned_full = pd.concat([aligned_features, aligned_masks.astype("int8")], axis=1)
aligned_full[label_col] = label_series

# Save
out_path = out_dir / "tabular_4_aligned_with_mask_and_label.csv"
aligned_full.to_csv(out_path, index=False)

print("8+8+1：", out_path)
print("shape：", aligned_full.shape)
aligned_full.head()

8+8+1： ../tabular_4/tabular_4_aligned_with_mask_and_label.csv
shape： (100000, 17)


,Age,Sex,BMI,GenHlth,HighBP,DiffWalk,HighChol,HeartDiseaseorAttack,Age_mask,Sex_mask,BMI_mask,GenHlth_mask,HighBP_mask,DiffWalk_mask,HighChol_mask,HeartDiseaseorAttack_mask,Diabetes_012
0,32.0,Female,27.32,0,0,0,0,0,1,1,1,0,1,0,0,1,0
1,29.0,Female,19.95,0,0,0,0,0,1,1,1,0,1,0,0,1,0
2,18.0,Male,23.76,0,0,0,0,0,1,1,1,0,1,0,0,1,0
3,41.0,Male,27.32,0,0,0,0,0,1,1,1,0,1,0,0,1,0
4,52.0,Female,23.75,0,0,0,0,0,1,1,1,0,1,0,0,1,0


In [21]:
from sklearn.model_selection import train_test_split
import pandas as pd
from pathlib import Path

data_path = Path("../tabular_4/tabular_4_aligned_with_mask_and_label.csv")
df = pd.read_csv(data_path)

# Parameter Settings
train_ratio = 0.7   
val_ratio   = 0.15  
test_ratio  = 0.15  
random_seed = 42    # Random seed for reproducibility

train_df, temp_df = train_test_split(df, test_size=(1 - train_ratio), random_state=random_seed, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=test_ratio / (test_ratio + val_ratio), random_state=random_seed)

out_dir = Path("../tabular_4")
train_path = out_dir / "tabular_4_train.csv"
val_path   = out_dir / "tabular_4_val.csv"
test_path  = out_dir / "tabular_4_test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Training set: {train_df.shape}, Validation set: {val_df.shape}, Test set: {test_df.shape}")
print(f"Save path:\n- {train_path}\n- {val_path}\n- {test_path}")


Training set: (69999, 17), Validation set: (15000, 17), Test set: (15001, 17)
Save path:
- ../tabular_4/tabular_4_train.csv
- ../tabular_4/tabular_4_val.csv
- ../tabular_4/tabular_4_test.csv
